# The three comparison baselines

This notebook runs the three baselines on the shared split and scores each one with our evaluation script.

1. Simple baselines: majority (one cluster per word, majority 0/1 label) and random (5 seeds)
2. Classical baseline: the Assignment 1 Baseline A method, re-run on the split
3. Pretrained baseline: frozen XLM-R (no fine-tuning) with APD as the change score

Before running: Runtime > Change runtime type > T4 GPU. The whole notebook takes about 10 minutes.
Every run is saved in `results/runs/<time>_baseline_<name>_seed<seed>/`.

In [ ]:
# Cell 1: get the repo and the packages, check the GPU
%cd /content
!rm -rf SenseShift
!git clone -q https://github.com/abiha25/SenseShift-Semantic-change-across-multiple-time-periods.git SenseShift
%cd /content/SenseShift
!pip install -q bcubed nltk pyyaml
!git log --oneline -1
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (the XLM-R part will be slow)')

Check that the scorer is still correct before we use it. The last line should say `31 passed`.

In [ ]:
# Cell 2: tests
!python -m pytest tests -q 2>&1 | tail -1

## 1. Simple baselines

Majority: one cluster per word, and the 0/1 label that is most common in the 9 non-test words.
Random: random clusters and random labels, run with the 5 seeds from the config (13, 42, 87, 1234, 2027).

Expected for majority: 1a `0.756`, 1b `0.000`, Subtask 2 `0.361`. Takes about 1 minute.

In [ ]:
# Cell 3: majority + random
!python -m src.baselines --which simple 2>&1 | grep -E "Evaluation:|1a  |1b  |2   macro|graded change|ERROR"


## 2. Classical baseline

Same features, clustering and classifier as Assignment 1 Baseline A. For every test word, the
settings are chosen on the other 9 words. The only change: the Subtask 2 threshold is now chosen
with the new official metric (macro F1).

Expected: 1a `0.697`, 1b `0.001` (the same as Assignment 1), Subtask 2 `0.430`. Takes about 1 to 2 minutes.

In [ ]:
# Cell 4: classical baseline
!python -m src.baselines --which classical

## 3. Pretrained baseline: frozen XLM-R + APD

XLM-R base, layer 8, mean of the target's sub-word vectors, nothing is trained.
Clustering of these vectors gives 1a and 1b. APD (average cosine distance between the usages of two
consecutive periods) gives a graded change score, which is checked against the gold change locally.
Subtask 2 uses the cosine between each usage and the sense definition.

The first run downloads the model (about 1 GB). Takes about 3 minutes on a T4.

In [ ]:
# Cell 5: frozen XLM-R + APD
!python -m src.baselines --which pretrained

## All baselines in one table

Saved as `results/baselines/summary.csv` and `summary.md`. The numbers in brackets are 95% confidence
intervals over the 10 words. A blank in the subgroup tables above means there is no ranking in that
group (all predicted change scores equal) or fewer than 5 period pairs.

In [ ]:
# Cell 6: summary table
!python -m src.baselines --table > /dev/null
from IPython.display import Markdown
Markdown(open('results/baselines/summary.md', encoding='utf-8').read())

## Download the results

This makes `baseline_results.zip` with the `results/runs` and `results/baselines` folders.
Do not upload the `cache` folder, it only holds the XLM-R vectors (large, and rebuilt automatically).

In [ ]:
# Cell 7: zip and download
!rm -f baseline_results.zip
!zip -qr baseline_results.zip results/runs results/baselines
from google.colab import files
files.download('baseline_results.zip')